In [ ]:
import sys
import pandas as pd

from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from src.utils.constants import DIAGNOSES_TO_CODES_PROCESSED_PATH

In [ ]:
def split_codes(codes, max_chars_per_row=20):
    if type(codes) == str:
        codes_list = codes.split(",")

        wrapped_codes_list = []
        current_string = ""
        i = 0
        while i < len(codes_list):
            if len(current_string) + len(codes_list[i]) > max_chars_per_row:
                wrapped_codes_list.append(current_string)
                current_string = codes_list[i]
            else:
                if len(current_string) > 0:
                    current_string += f",{codes_list[i]}"
                else:
                    current_string += f"{codes_list[i]}"

            i += 1

        if current_string != "":
            wrapped_codes_list.append(current_string)

        return wrapped_codes_list
    else:
        return [""]


def split_text(text, max_chars_per_row=20):
    words = text.split()

    wrapped_words_list = []
    current_string = ""
    i = 0
    while i < len(words):
        if len(current_string) + len(words[i]) > max_chars_per_row:
            wrapped_words_list.append(" " + current_string)
            current_string = words[i]
        else:
            if len(current_string) > 0:
                current_string += f" {words[i]}"
            else:
                current_string += f"{words[i]}"

        i += 1

    if current_string != "":
        wrapped_words_list.append(current_string)

    return wrapped_words_list


def preprocess_table(df, max_chars_per_row=20, max_chars_per_row_diagnosis=20):
    rows = []
    for _, row in df.iterrows():
        diag_split = split_text(row["Diagnosis"], max_chars_per_row_diagnosis)
        icd10_split = split_codes(row["ICD-10"], max_chars_per_row)
        icd9_split = split_codes(row["ICD-9"], max_chars_per_row)
        max_len = max(len(diag_split), len(icd10_split), len(icd9_split))

        for i in range(max_len):
            diagnosis = diag_split[i] if i < len(diag_split) else ""
            icd10 = icd10_split[i] if i < len(icd10_split) else ""
            icd9 = icd9_split[i] if i < len(icd9_split) else ""

            start_row = True if i == 0 else False
            rows.append([diagnosis, icd10, icd9, start_row])

        rows.append(["", "", "", ""])

    return pd.DataFrame(rows, columns=list(df.columns) + ["Start Row"])


def remove_duplicate_codes(codes):
    # Split the string into a list, convert to a set to remove duplicates, then join back into a string
    try:
        return ",".join(sorted(set(codes.split(","))))
    except:
        return codes

In [ ]:
df_codes = pd.read_csv(DIAGNOSES_TO_CODES_PROCESSED_PATH)
df_codes = df_codes[["Diagnosis", "Processed Codes ICD10", "Processed Codes ICD9"]]
df_codes = df_codes.rename(
    columns={"Processed Codes ICD10": "ICD-10", "Processed Codes ICD9": "ICD-9"}
)
df_codes["Diagnosis"] = df_codes["Diagnosis"].str.replace("_", " ")

In [ ]:
# Deduplicate codes
df_codes["ICD-10"] = df_codes["ICD-10"].apply(remove_duplicate_codes)
df_codes["ICD-9"] = df_codes["ICD-9"].apply(remove_duplicate_codes)

In [ ]:
df_codes_processed = preprocess_table(
    df_codes, max_chars_per_row=60, max_chars_per_row_diagnosis=20
)

In [ ]:
sub_dfs = [df_codes_processed]

In [ ]:
print(len(sub_dfs))

In [ ]:
beginning = """
\\afterpage{
\\begin{tiny}
\\centering
"""
caption = """
\\caption{A comprehensive mapping of diagnosis groups used in the experiments to their corresponding ICD-10 and ICD-9 codes.}\\\\"""
continued_caption = """
\\caption[]{(cont'd) A comprehensive mapping of diagnosis groups used in the experiments to their corresponding ICD-10 and ICD-9 codes.} \\\\"""
ending = """\\label{tab:icd-table}
\\end{longtable}
\\end{tiny}
}
"""

for i, df in enumerate(sub_dfs):
    latex_string = beginning + df[["Diagnosis", "ICD-10", "ICD-9"]].to_latex(
        index=False, longtable=True, escape=False
    )
    latex_string = latex_string.replace("lll", "r|r|r")
    latex_string = latex_string.replace(
        "\\begin{longtable}{r|r|r}", "\\begin{longtable}{r|r|r}" + caption
    )

    latex_string = latex_string.replace(
        "\\endfirsthead", "\\endfirsthead" + continued_caption
    )

    latex_string = latex_string.replace("\\end{longtable}", ending)
    print(latex_string)

    break